In [2]:
import pandas as pd
import pickle
import numpy as np
import torch
from sklearn.neighbors import NearestNeighbors
from torch_geometric.data import HeteroData
import torch.nn as nn
from torch_geometric.nn import HeteroConv, ChebConv
import plotly.graph_objects as go
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.sparse as sp
import torch.nn.functional as F

c:\Users\lucch\Desktop\thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ==========================================
# SECTION 1: IMPORTS & INITIAL RAW DATA LOADING
# ==========================================
with open("processed_data_pkl/google_master.pkl", "rb") as f:
    google = pickle.load(f)

with open("processed_data_pkl/pollution_dic.pkl", "rb") as f:
    pollution = pickle.load(f)

with open("processed_data_pkl/processed_traffic_data.pkl", "rb") as f:
    traffic = pickle.load(f)

In [4]:
for site_ref, directions in google.items():
    for direction, df in directions.items():
        agg_dict = {"Count": "sum"}
        for avg_col in ["speed(mph)", "miles", "avtime", "TravelTime"]:
            if avg_col in df.columns:
                agg_dict[avg_col] = "mean"

        google[site_ref][direction] = (
            df
            .assign(timestamp=df['timestamp'].dt.floor('1h'))
            .groupby('timestamp', as_index=False)
            .agg(agg_dict)
        )


In [5]:
for site_ref, directions in traffic.items():
    for direction, df in directions.items():
        if site_ref in google and direction in google[site_ref]:
            add_df = google[site_ref][direction][["timestamp"] + [c for c in ["speed(mph)", "miles", "avtime", "TravelTime"] if c in google[site_ref][direction].columns]].copy()
            if not add_df.empty:
                traffic[site_ref][direction] = (
                    df.merge(add_df, on="timestamp", how="left")
                )


In [6]:
google_speed = pd.read_csv("classified_datasets/traffic/google_speed/google1.csv")
google_speed["section"] = google_speed["section"].apply(lambda x: x.split("-")[1]).astype(int)
google_speed

,section,timestamp,miles,avtime,TravelTime,speed(mph)
0,1,2017-07-02 22:48:12,9.411907,0.457222,0.389444,24.167520
1,2,2017-07-02 22:48:13,4.449638,0.236944,0.201111,22.125270
2,3,2017-07-02 22:48:13,5.587989,0.411667,0.339167,16.475644
3,4,2017-07-02 22:48:14,4.315422,0.230833,0.200556,21.517338
4,5,2017-07-02 22:48:14,4.611816,0.309444,0.251389,18.345344
...,...,...,...,...,...,...
552028,17,2018-04-18 10:30:10,11.891177,0.498056,0.500000,23.782354
552029,18,2018-04-18 10:30:10,10.619852,0.496389,0.488889,21.722424
552030,19,2018-04-18 10:30:11,11.999295,0.531667,0.538333,22.289713
552031,20,2018-04-18 10:30:11,19.283628,0.695278,0.662778,29.095163


In [7]:
google_speed["timestamp"] = pd.to_datetime(google_speed["timestamp"])
google_speed = google_speed.sort_values(["section", "timestamp"]).reset_index(drop=True)

google_speed_sections = {}

for section, group in google_speed.groupby("section", sort=True):
    group = group.sort_values("timestamp").reset_index(drop=True)
    hour_index = pd.date_range(
        start=group["timestamp"].min().floor("h"),
        end=group["timestamp"].max().ceil("h"),
        freq="h"
    )
    targets = pd.DataFrame({"timestamp": hour_index})
    nearest_hourly = pd.merge_asof(
        targets,
        group,
        on="timestamp",
        direction="nearest"
    )
    google_speed_sections[section] = nearest_hourly

google_speed_sections

{1:                timestamp  section      miles    avtime  TravelTime  speed(mph)
 0    2017-07-02 22:00:00        1   9.411907  0.457222    0.389444   24.167520
 1    2017-07-02 23:00:00        1   9.411907  0.457222    0.388889   24.202045
 2    2017-07-03 00:00:00        1   9.411907  0.457222    0.377778   24.913870
 3    2017-07-03 01:00:00        1   9.411907  0.457222    0.389722   24.150295
 4    2017-07-03 02:00:00        1   9.411907  0.457222    0.394444   23.861172
 ...                  ...      ...        ...       ...         ...         ...
 6945 2018-04-18 07:00:00        1   9.420606  0.470000    0.428889   21.965143
 6946 2018-04-18 08:00:00        1   9.411907  0.474167    0.586944   16.035430
 6947 2018-04-18 09:00:00        1   9.418120  0.508056    0.590833   15.940401
 6948 2018-04-18 10:00:00        1  10.663348  0.479722    0.483056   22.074785
 6949 2018-04-18 11:00:00        1  10.663348  0.479722    0.487500   21.873534
 
 [6950 rows x 6 columns],
 2:      

In [8]:
for sensor, directions in traffic.items():
    for direction, df in directions.items():
        cols_to_drop = [c for c in df.columns if c in ["speed(mph)", "miles", "avtime", "TravelTime"]]
        df = df.drop(columns=cols_to_drop, errors="ignore")

        if df.empty:
            continue

        route = df.loc[0, "route"]
        if route not in google_speed_sections:
            continue

        gdf = google_speed_sections[route]

        # --- Build full timestamp index (correct way) ---
        full_index = pd.Index(df["timestamp"]).union(pd.Index(gdf["timestamp"]))

        # --- Expand traffic DF to full timeline ---
        expanded = (
            df.set_index("timestamp")
              .reindex(full_index)
              .reset_index()
              .rename(columns={"index": "timestamp"})
        )

        # --- Merge Google speed data cleanly ---
        merged = expanded.merge(
            gdf[["timestamp", "speed(mph)", "miles", "avtime", "TravelTime"]],
            on="timestamp",
            how="left"
        )

        traffic[sensor][direction] = merged

In [9]:
master = pd.Index(traffic[2]["N"]["timestamp"])
for sensor, directions in traffic.items():
    for direction, df in directions.items():
        if len(df.columns)<10:
            continue
        # Reindex to the master timeline
        expanded = (
            df.set_index("timestamp")
              .reindex(master)
              .reset_index()
              .rename(columns={"index": "timestamp"})
        )

        traffic[sensor][direction] = expanded

In [10]:
for direction, df_dict in traffic.items():
    for df_id, df in df_dict.items():
        df["weekday"] = df["timestamp"].dt.weekday

        num_unique = df.nunique(dropna=True)
        single_val_cols = num_unique[num_unique == 1].index
        for col in single_val_cols:
            first_valid = df[col].dropna().iloc[0]
            df[col] = df[col].fillna(first_valid)

        # Rich temporal features — critical for learning daily traffic cycles
        df["hour"]         = df["timestamp"].dt.hour.astype(np.float32)
        df["hour_sin"]     = np.sin(2 * np.pi * df["hour"] / 24).astype(np.float32)
        df["hour_cos"]     = np.cos(2 * np.pi * df["hour"] / 24).astype(np.float32)
        df["weekday_sin"]  = np.sin(2 * np.pi * df["weekday"] / 7).astype(np.float32)
        df["weekday_cos"]  = np.cos(2 * np.pi * df["weekday"] / 7).astype(np.float32)
        df["is_weekend"]   = (df["weekday"] >= 5).astype(np.float32)


In [11]:
training_data = {}
no_speed = {}
for direction, df_dict in traffic.items():
    for df_id, df in df_dict.items():
        if len(df.columns) > 16:
            if direction not in training_data:
                training_data[direction] = {}
            training_data[direction][df_id] = df
        else:
            if direction not in no_speed:
                no_speed[direction] = {}
            no_speed[direction][df_id] = df


In [12]:
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler


def knn_impute_count(
    df,
    k=5
):
    """
    Impute missing traffic counts using KNN with:
    - temporal features
    - speed
    - travel time ratio

    Uses:
    - feature standardization
    - distance-weighted averaging
    """
    df_imputed = df.copy()

    df_imputed['count_imputed'] = False

    df_imputed['travel_time_ratio'] = df_imputed['TravelTime'] /df_imputed['avtime'].replace(0, np.nan)
    df_imputed['travel_time_ratio'] = df_imputed['travel_time_ratio'].replace([np.inf, -np.inf], np.nan).fillna(1.0)

    # --------------------------------------------------
    # Features
    # --------------------------------------------------
    feature_cols = ['hour_sin','hour_cos','weekday_sin','weekday_cos','is_weekend']

    if 'speed(mph)' in df_imputed.columns:
        feature_cols.append('speed(mph)')

    feature_cols.append('travel_time_ratio')

    # Fill feature NaNs
    for col in feature_cols:
        if df_imputed[col].isna().any():
            df_imputed[col] = df_imputed[col].fillna(df_imputed[col].median())

    # --------------------------------------------------
    # Missing masks
    # --------------------------------------------------
    missing_mask = df_imputed['count'].isna()
    non_missing_mask = ~missing_mask

    if not missing_mask.any():
        return df_imputed

    if non_missing_mask.sum() < k:
        print(f"Warning: only {non_missing_mask.sum()} "f"non-missing values available.")
        return df_imputed

    # --------------------------------------------------
    # Build KNN dataset
    # --------------------------------------------------
    X_train = df_imputed.loc[non_missing_mask,feature_cols].values

    y_train = df_imputed.loc[non_missing_mask,'count'].values

    X_missing = df_imputed.loc[missing_mask,feature_cols].values

    # --------------------------------------------------
    # Standardize features
    # --------------------------------------------------
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_missing_scaled = scaler.transform(X_missing)

    # --------------------------------------------------
    # Reduce influence of speed-related features
    # --------------------------------------------------
    feature_weight = 0.4  # try 0.2-0.5
    if 'speed(mph)' in feature_cols:
        speed_idx = feature_cols.index('speed(mph)')

        X_train_scaled[:, speed_idx] *= feature_weight
        X_missing_scaled[:, speed_idx] *= feature_weight

    if 'travel_time_ratio' in feature_cols:
        ratio_idx = feature_cols.index('travel_time_ratio')

        X_train_scaled[:, ratio_idx] *= feature_weight
        X_missing_scaled[:, ratio_idx] *= feature_weight

    # --------------------------------------------------
    # KNN
    # --------------------------------------------------
    knn = NearestNeighbors(n_neighbors=min(k, len(X_train_scaled)))
    knn.fit(X_train_scaled)
    distances, indices = knn.kneighbors(X_missing_scaled)

    # --------------------------------------------------
    # Distance-weighted imputation
    # --------------------------------------------------
    imputed_values = []

    for i, neighbor_idx in enumerate(indices):
        neighbor_counts = y_train[neighbor_idx]
        weights = 1.0 / (distances[i] + 1e-6)
        weighted_count = np.average(neighbor_counts,weights=weights)
        imputed_values.append(int(round(weighted_count)))

    # --------------------------------------------------
    # Apply imputed values
    # --------------------------------------------------
    df_imputed.loc[missing_mask,'count'] = imputed_values
    df_imputed.loc[missing_mask,'count_imputed'] = True

    return df_imputed

# Apply imputation and speed adjustment to all sensors
imputed_training_data = {}

print("Applying KNN imputation with speed-based adjustment...")
total_processed = 0
total_imputed = 0

for sensor_id, directions in training_data.items():
    imputed_training_data[sensor_id] = {}
    
    for direction, df in directions.items():
        original_missing = df['count'].isna().sum()
        
        # Apply KNN imputation + speed adjustment
        df_processed = knn_impute_count(df, k=5)        
        imputed_training_data[sensor_id][direction] = df_processed
        
        total_processed += 1
        if original_missing > 0:
            total_imputed += original_missing
            print(f"✓ Sensor {sensor_id}, Direction {direction}: Imputed {original_missing} values | Speed adjustments applied")

print(f"\n{'='*60}")
print(f"KNN imputation complete!")
print(f"Total sensors processed: {total_processed}")
print(f"Total values imputed: {total_imputed}")
print(f"{'='*60}")

Applying KNN imputation with speed-based adjustment...
✓ Sensor 1, Direction N: Imputed 6614 values | Speed adjustments applied
✓ Sensor 1, Direction S: Imputed 6614 values | Speed adjustments applied
✓ Sensor 2, Direction N: Imputed 6614 values | Speed adjustments applied
✓ Sensor 2, Direction S: Imputed 6614 values | Speed adjustments applied
✓ Sensor 3, Direction E: Imputed 6614 values | Speed adjustments applied
✓ Sensor 3, Direction W: Imputed 6614 values | Speed adjustments applied
✓ Sensor 4, Direction N: Imputed 6614 values | Speed adjustments applied
✓ Sensor 4, Direction S: Imputed 6614 values | Speed adjustments applied
✓ Sensor 5, Direction E: Imputed 6614 values | Speed adjustments applied
✓ Sensor 5, Direction W: Imputed 6614 values | Speed adjustments applied
✓ Sensor 6, Direction N: Imputed 6614 values | Speed adjustments applied
✓ Sensor 6, Direction S: Imputed 6614 values | Speed adjustments applied
✓ Sensor 7, Direction E: Imputed 6614 values | Speed adjustments appl

In [28]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

# ==========================================
# INTERACTIVE VISUALIZATION: Plot one sensor-direction
# ==========================================

viz_direction = 'E'
viz_sensor = 5

if viz_sensor in imputed_training_data and viz_direction in imputed_training_data[viz_sensor]:

    df_viz = imputed_training_data[viz_sensor][viz_direction]
    df_original = training_data[viz_sensor][viz_direction]

    # Identify missing values in original data
    imputed_mask = df_original['count'].isna()

    # Create a series containing ONLY imputed values
    imputed_line = np.full(len(df_viz), np.nan)
    imputed_line[imputed_mask] = df_viz.loc[imputed_mask, 'count']

    # Create figure with secondary y-axis
    fig = make_subplots(rows=1, cols=1, specs=[[{"secondary_y": True}]])

    # --------------------------------------------------
    # Observed data (blue line with gaps)
    # --------------------------------------------------
    fig.add_trace(
        go.Scatter(
            x=df_original['timestamp'],
            y=df_original['count'],
            mode='lines',
            name='Observed', line=dict(color='royalblue', width=1.5),
            connectgaps=False,
            hovertemplate=('<b>Observed</b><br>''Time: %{x}<br>''Count: %{y}<extra></extra>')), 
        secondary_y=False)
    # --------------------------------------------------
    # Imputed regions only (red line)
    # --------------------------------------------------
    fig.add_trace(
        go.Scatter(
            x=df_viz['timestamp'],
            y=imputed_line,
            mode='lines',
            name='KNN Imputed',
            line=dict(color='red',width=1.5),
            connectgaps=False,
            hovertemplate=('<b>KNN Imputed</b><br>''Time: %{x}<br>''Count: %{y}<extra></extra>')),
        secondary_y=False)
    # --------------------------------------------------
    # Travel Time Ratio
    # --------------------------------------------------
    if 'travel_time_ratio' in df_viz.columns:
        fig.add_trace(
            go.Scatter(
                x=df_viz['timestamp'],
                y=df_viz['travel_time_ratio'],
                mode='lines',
                name='Travel Time Ratio',
                line=dict(color='orange', width=0.7,dash='dot'),
                opacity=0.6,
                hovertemplate=('<b>Travel Time Ratio</b><br>''Time: %{x}<br>''Ratio: %{y:.3f}<extra></extra>')),
            secondary_y=True)
        
    fig.update_xaxes(
        title_text="Timestamp",
        rangeslider_visible=True,
        rangeselector=dict(
            buttons=[
                dict(count=7, label="1W", step="day"),
                dict(count=1, label="1M", step="month"),
                dict(count=3, label="3M", step="month"),
                dict(step="all")]))

    fig.update_yaxes(title_text="Traffic Count",secondary_y=False)
    fig.update_yaxes(title_text="Travel Time Ratio",secondary_y=True)

    fig.update_layout(
        title=f'KNN Imputation - Sensor {viz_sensor}, Direction {viz_direction}',
        hovermode='x unified',
        template='plotly_white',
        height=700,
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bgcolor="rgba(255,255,255,0.8)"))

    fig.show()

In [17]:
for sensor_id, directions in imputed_training_data.items():
    for direction, df in directions.items():
        df.drop(columns=["route", "weekday", "traffic_intensity", "distance_to_center_km", "weekday_sin", "weekday_cos", "travel_time_ratio"], inplace=True, errors="ignore")

In [29]:
with open("processed_data_pkl/imputed_training_data.pkl", "wb") as f:
    pickle.dump(imputed_training_data, f)